# Skenario A: Visualisasi REST vs gRPC

Notebook ini membaca `results/scenario_a/raw.csv` langsung dari GitHub (jadi Anda cukup **Run all**
tiap kali data baru sudah di-push, tidak perlu upload manual) dan menghasilkan 4 figure: **Latency**,
**Throughput**, **CPU**, dan **RAM**, masing-masing dipecah per segmen komunikasi.

Setiap panel memakai gaya P50 garis solid, P99 garis putus-putus, dengan area terarsir di antaranya
supaya rentang persentil kelihatan sekali pandang. Warna REST/gRPC sudah divalidasi CVD-safe
(`node scripts/validate_palette.js` dari skill dataviz, ΔE 74.6, jauh di atas ambang 12).

Mau ubah judul, warna, atau ukuran figure? Semuanya ada di sel **Konfigurasi** di bawah. Sel-sel
setelahnya tidak perlu disentuh.

## Konfigurasi

Ubah nilai di sel ini sesuai kebutuhan, lalu Run all.

In [ ]:
# ==== Konfigurasi: ubah di sini ====

# URL raw.csv di GitHub (branch dev). Kalau gagal diakses (mis. belum di-push,
# atau repo private), notebook otomatis fallback ke file lokal di bawah.
GITHUB_RAW_CSV_URL = (
    "https://raw.githubusercontent.com/afifksupriyadi/"
    "grpc-rest-benchmark-video-transcoder/dev/results/scenario_a/raw.csv"
)
LOCAL_CSV_FALLBACK = "../results/scenario_a/raw.csv"

# Warna REST/gRPC. Pasangan biru/merah ini sudah divalidasi CVD-safe (skill dataviz).
# Konsisten dengan warna yang sudah dipakai di tabel HTML (scripts/collect_scenario_a.py).
COLORS = {
    "rest": "#2a78d6",
    "grpc": "#e34948",
}
PROTOCOL_LABEL = {"rest": "REST", "grpc": "gRPC"}

TITLES = {
    "latency_seconds": "Skenario A: LATENCY REST vs gRPC",
    "throughput_bytes_per_second": "Skenario A: THROUGHPUT REST vs gRPC",
    "cpu_usage_ratio": "Skenario A: CPU REST vs gRPC",
    "memory_usage_bytes": "Skenario A: RAM REST vs gRPC",
}

FIGSIZE_PER_PANEL = (5, 5.5)  # (lebar, tinggi) per subplot segmen, dalam inci
FONT_SIZES = {
    "suptitle": 18,
    "subplot_title": 14,
    "axis_label": 12,
    "tick_label": 11,
    "legend": 12,
}
OUTPUT_DIR = "figures"  # folder PNG hasil export, dibuat relatif ke lokasi notebook
DPI = 200

## Setup (tidak perlu diubah)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

try:
    df = pd.read_csv(GITHUB_RAW_CSV_URL)
    print(f"Data dimuat dari GitHub: {len(df)} baris")
except Exception as e:
    print(f"Gagal fetch dari GitHub ({e}), pakai file lokal: {LOCAL_CSV_FALLBACK}")
    df = pd.read_csv(LOCAL_CSV_FALLBACK)
    print(f"Data dimuat dari lokal: {len(df)} baris")

df["payload_mb"] = df["payload_size"].str.replace("mb", "", case=False, regex=False).astype(int)
df = df.sort_values("payload_mb")
df.head()

In [ ]:
SEGMENT_LABEL = {
    "client_to_gateway": "Client ke Gateway",
    "gateway_to_worker": "Gateway ke Worker",
    "worker_to_gateway": "Worker ke Gateway",
    "gateway_to_client": "Gateway ke Client",
}
SEGMENT_ORDER = list(SEGMENT_LABEL.keys())

# Chrome tokens (skill dataviz: gridlines hairline recessive, teks pakai warna
# ink/muted, bukan warna seri).
GRID_COLOR = "#e1e0d9"
AXIS_COLOR = "#c3c2b7"
MUTED_TEXT = "#898781"


def plot_metric_figure(metric, title):
    """One figure, one subplot per segment that actually has data for this
    metric (CPU/RAM naturally drop 'Worker ke Gateway'; see architecture note
    in the generated HTML tables). P50 solid + P99 dashed + shaded band,
    REST/gRPC as the only two colors, legend shared once for the whole figure."""
    sub = df[df["metric"] == metric]
    segments = [s for s in SEGMENT_ORDER if s in sub["segment"].unique()]
    unit = sub["unit"].mode().iat[0]

    fig, axes = plt.subplots(
        1, len(segments),
        figsize=(FIGSIZE_PER_PANEL[0] * len(segments), FIGSIZE_PER_PANEL[1]),
        squeeze=False,
    )
    axes = axes[0]

    legend_handles = {}
    for ax, seg in zip(axes, segments):
        seg_df = sub[sub["segment"] == seg]
        for protocol, color in COLORS.items():
            pdf = seg_df[seg_df["protocol"] == protocol].sort_values("payload_mb")
            p50 = pdf[pdf["percentile"] == 50]
            p99 = pdf[pdf["percentile"] == 99]
            if p50.empty:
                continue
            label = f"{PROTOCOL_LABEL[protocol]} P50"
            (line,) = ax.plot(
                p50["payload_mb"], p50["value"],
                color=color, lw=2.5, marker="o", markersize=10, label=label,
            )
            legend_handles[label] = line
            if not p99.empty:
                label99 = f"{PROTOCOL_LABEL[protocol]} P99"
                (line99,) = ax.plot(
                    p99["payload_mb"], p99["value"],
                    color=color, lw=2.5, ls="--", marker="s", markersize=10, label=label99,
                )
                legend_handles[label99] = line99
                ax.fill_between(
                    p50["payload_mb"], p50["value"], p99["value"],
                    color=color, alpha=0.10,
                )

        ax.set_title(SEGMENT_LABEL[seg], fontsize=FONT_SIZES["subplot_title"])
        ax.set_xlabel("Ukuran Payload (MB)", fontsize=FONT_SIZES["axis_label"], color=MUTED_TEXT)
        ax.set_ylabel(unit, fontsize=FONT_SIZES["axis_label"], color=MUTED_TEXT)
        ax.set_xticks(sorted(df["payload_mb"].unique()))
        ax.grid(axis="y", color=GRID_COLOR, lw=1)
        ax.set_axisbelow(True)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.spines["left"].set_color(AXIS_COLOR)
        ax.spines["bottom"].set_color(AXIS_COLOR)
        ax.tick_params(colors=MUTED_TEXT, labelsize=FONT_SIZES["tick_label"])

    fig.legend(
        legend_handles.values(), legend_handles.keys(),
        loc="lower center", ncol=len(legend_handles), frameon=False,
        bbox_to_anchor=(0.5, -0.08), fontsize=FONT_SIZES["legend"],
    )
    fig.suptitle(title, fontsize=FONT_SIZES["suptitle"], fontweight="bold", y=1.04)
    fig.tight_layout()
    return fig

## Latency

In [ ]:
Path(OUTPUT_DIR).mkdir(exist_ok=True)
fig = plot_metric_figure("latency_seconds", TITLES["latency_seconds"])
fig.savefig(f"{OUTPUT_DIR}/scenario_a_latency.png", dpi=DPI, bbox_inches="tight")
plt.show()

## Throughput

In [ ]:
fig = plot_metric_figure("throughput_bytes_per_second", TITLES["throughput_bytes_per_second"])
fig.savefig(f"{OUTPUT_DIR}/scenario_a_throughput.png", dpi=DPI, bbox_inches="tight")
plt.show()

## CPU

In [ ]:
fig = plot_metric_figure("cpu_usage_ratio", TITLES["cpu_usage_ratio"])
fig.savefig(f"{OUTPUT_DIR}/scenario_a_cpu.png", dpi=DPI, bbox_inches="tight")
plt.show()

## RAM

In [ ]:
fig = plot_metric_figure("memory_usage_bytes", TITLES["memory_usage_bytes"])
fig.savefig(f"{OUTPUT_DIR}/scenario_a_ram.png", dpi=DPI, bbox_inches="tight")
plt.show()

---
Selesai. 4 file PNG ada di folder `figures/` (di Colab: klik ikon folder di sidebar kiri untuk
download). Ubah warna/judul di sel **Konfigurasi** paling atas lalu **Run all** lagi kalau mau
tweak tanpa sentuh kode plotting.